# 03. XAI Concepts: TCAV, Concept Bottleneck Models, DiCE Counterfactuals, and Global ALE

- **Category:** 10_xai_and_ai_analysis_methods
- **Lineage & Citations:** Wachter et al. (2017 *Counterfactual Explanations Without Opening the Black Box*); Kim et al. (2018 *Interpretability Beyond Feature Attribution: Quantitative Testing with Concept Activation Vectors TCAV*); Koh et al. (2020 *Concept Bottleneck Models CBM*); Apley & Zhu (2020 *Visualizing the Effects of Predictor Variables in Black Box Models: ALE*); Mothilal et al. (2020 *Explaining Machine Learning Classifiers through Diverse Counterfactual Explanations DiCE*).
- **Purpose:** Implement Concept Activation Vectors (TCAV), test-time expert concept intervention on Concept Bottleneck Models (CBM), Determinantal Point Process counterfactual diversity (DiCE), and Accumulated Local Effects (ALE).
- **Topic Connections:** `🔗 Upstream Prerequisites:` [02. XAI Gradient Propagation IG and LRP](02_xai_gradient_propagation_ig_lrp_deeplift_gradcam_and_attention.ipynb) $\to$ `🔗 Downstream Syntheses:` [04. XAI Data Attribution Influence Functions TracIn TRAK](04_xai_data_attribution_influence_functions_tracin_and_trak.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Avoid unconstrained continuous concept bottlenecks where encoders leak non-concept information into continuous latent channels (rendering test-time intervention ineffective); never use Partial Dependence Plots (PDP) on strongly correlated features where off-manifold evaluations yield severe extrapolation bias (use ALE instead).

## Mathematical Foundations & Governing Equations

1. **TCAV Directional Sensitivity**: $S_{C, k}(x) = \nabla_{a} f_k(x) \cdot \frac{v_C}{\|v_C\|_2}, \quad \text{TCAV}_{C, k} = \frac{1}{N} \sum_{i=1}^N \mathbb{I}(S_{C, k}(x_i) > 0)$
2. **CBM Test-Time Intervention**: $c^{\text{int}} = (1 - m) \odot \hat{c} + m \odot c^{\text{true}}, \quad \hat{y} = c^{\text{int}} W_{\text{task}} + b_{\text{task}}$
3. **DiCE DPP Diversity**: $K_{ij} = \frac{1}{1 + \|c_i - c_j\|_2}, \quad \text{Diversity} = \det(K)$
4. **Accumulated Local Effects (ALE)**: $\text{ALE}(x_k) = \sum_{m=1}^k \Delta_m - \frac{1}{K}\sum_{k=1}^K \sum_{m=1}^k \Delta_m$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/10_xai_and_ai_analysis_methods/03_xai_concepts_tcav_cbm_counterfactuals_dice_and_global_ale.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def calc_tcav(grads, cav):
    cav_norm = cav / torch.sqrt((cav * cav).sum() + 1e-8)
    derivs = grads @ cav_norm
    score = torch.where(derivs > 0.0, 1.0, 0.0).mean()
    return derivs, score

def calc_cbm_intervention(pred_c, true_c, mask, w, b):
    c_int = (1.0 - mask) * pred_c + mask * true_c
    return c_int, c_int @ w + b

def calc_dice_diversity(cfs):
    diff = cfs.unsqueeze(1) - cfs.unsqueeze(0)
    dist = torch.sqrt((diff * diff).sum(dim=-1) + 1e-8)
    return torch.linalg.det(1.0 / (1.0 + dist))

def calc_ale(diffs):
    cumsum = torch.cumsum(diffs, dim=0)
    return cumsum - cumsum.mean()

In [ ]:
# Parity verification against golden reference tensors
derivs, score = calc_tcav(weights["tcav_grads"], weights["tcav_cav"])
assert torch.max(torch.abs(derivs - weights["tcav_derivs_ref"])) < 1e-5, "TCAV derivs mismatch"
assert torch.max(torch.abs(score - weights["tcav_score_ref"])) < 1e-5, "TCAV score mismatch"

c_int, y_pred = calc_cbm_intervention(
    weights["cbm_pred_c"], weights["cbm_true_c"], weights["cbm_mask"], weights["cbm_w"], weights["cbm_b"]
)
assert torch.max(torch.abs(c_int - weights["cbm_c_int_ref"])) < 1e-5, "CBM c_int mismatch"
assert torch.max(torch.abs(y_pred - weights["cbm_y_ref"])) < 1e-5, "CBM y_pred mismatch"

div = calc_dice_diversity(weights["dice_cfs"])
assert torch.max(torch.abs(div - weights["dice_div_ref"])) < 1e-5, "DiCE diversity mismatch"

ale = calc_ale(weights["ale_diffs"])
assert torch.max(torch.abs(ale - weights["ale_ref"])) < 1e-5, "ALE mismatch"
print("All TCAV, CBM, DiCE, and ALE assertions passed!")

In [ ]:
# Benchmarking concept attribution and counterfactual diversity operators
t0 = time.perf_counter()
for _ in range(500):
    _ = calc_tcav(weights["tcav_grads"], weights["tcav_cav"])
    _ = calc_cbm_intervention(weights["cbm_pred_c"], weights["cbm_true_c"], weights["cbm_mask"], weights["cbm_w"], weights["cbm_b"])
    _ = calc_dice_diversity(weights["dice_cfs"])
    _ = calc_ale(weights["ale_diffs"])
lat_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"Fused TCAV + CBM + DiCE latency: {lat_us:.2f} us")

In [ ]:
# Failure Mode: Counterfactual Collinearity & Redundancy Collapse
# High diversity: distinct counterfactual options give genuine alternative recourse
cfs_diverse = weights["dice_cfs"]
div_high = calc_dice_diversity(cfs_diverse).item()
print(f"Diverse counterfactual DPP volume: {div_high:.4f}")
assert div_high > 0.5, "Diverse set must span non-degenerate volume!"
# Redundant counterfactuals: almost identical points produce determinant near zero
cfs_redundant = cfs_diverse.clone()
cfs_redundant[1] = cfs_redundant[0] + 0.001
div_low = calc_dice_diversity(cfs_redundant).item()
print(f"Redundant counterfactual DPP volume: {div_low:.6f}")
assert div_low < 0.01, "Determinant collapses on redundant counterfactuals!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **TCAV** probes model representations using user-defined high-level concepts rather than low-level features.
2. **Concept Bottleneck Models (CBM)** enable test-time human expert intervention by exposing explicit intermediate concepts.
3. **DiCE** enforces diversity among counterfactual explanations via DPP kernel determinants to avoid redundant recourse.
4. **ALE** eliminates the severe extrapolation bias of Partial Dependence Plots on correlated features.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [03_xai_concepts_tcav_cbm_counterfactuals_dice_and_global_ale.md](../../markdown/10_xai_and_ai_analysis_methods/03_xai_concepts_tcav_cbm_counterfactuals_dice_and_global_ale.md)
- **PyTorch 2.x Module:** [03_xai_concepts_tcav_cbm_counterfactuals_dice_and_global_ale.py](../../pytorch2/10_xai_and_ai_analysis_methods/03_xai_concepts_tcav_cbm_counterfactuals_dice_and_global_ale.py)
- **Functional JAX Module:** [03_xai_concepts_tcav_cbm_counterfactuals_dice_and_global_ale.py](../../jax/10_xai_and_ai_analysis_methods/03_xai_concepts_tcav_cbm_counterfactuals_dice_and_global_ale.py)
- **Native Rust Candle Track:** [03_xai_concepts_tcav_cbm_counterfactuals_dice_and_global_ale.rs](../../candle/10_xai_and_ai_analysis_methods/03_xai_concepts_tcav_cbm_counterfactuals_dice_and_global_ale.rs)
- **Hardware Kernel (CPU / WGSL):** [03_xai_concepts_tcav_cbm_counterfactuals_dice_and_global_ale.rs](../../kernels/10_xai_and_ai_analysis_methods/03_xai_concepts_tcav_cbm_counterfactuals_dice_and_global_ale.rs)
- **Golden SafeTensors:** [03_xai_concepts_tcav_cbm_counterfactuals_dice_and_global_ale.safetensors](../../weights/10_xai_and_ai_analysis_methods/03_xai_concepts_tcav_cbm_counterfactuals_dice_and_global_ale.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 02. XAI Gradient Propagation IG LRP](02_xai_gradient_propagation_ig_lrp_deeplift_gradcam_and_attention.ipynb) | [Master Index](../../README.md#10_xai_and_ai_analysis_methods) | [Next: 04. XAI Data Attribution Influence Functions TracIn TRAK](04_xai_data_attribution_influence_functions_tracin_and_trak.ipynb) &rarr;